# Examen Unidad 5 — Sistema de Inteligencia Visual ESG

**Caso:** Atlas Global Capital | Análisis ESG y Riesgo de Cadena de Suministro  
**Materia:** Programación Avanzada para Ciencia de Datos  
**Carrera:** Ingeniería en Ciencia de Datos — Instituto Tecnológico de Tláhuac III  
**Docente:** Alexsander Pedraza Bonilla  
**Alumno:** Yair García  

---

Este notebook consolida las 4 misiones del examen sobre un dataset multinivel de 150 países × 24 años (2000–2023). El dashboard interactivo equivalente está en `app.py` (Streamlit).

## 0. Carga del dataset y exploración inicial

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from scipy.cluster.hierarchy import linkage, fcluster

df = pd.read_csv('global_esg_risk.csv')
print(f'Shape: {df.shape}')
print(f'Países: {df["ISO_Code"].nunique()} | Años: {df["Year"].nunique()} ({df["Year"].min()}-{df["Year"].max()})')
df.head()

In [ ]:
df.describe().round(2)

## Misión 1 — Diagnóstico Estadístico

### 1.1 KDE Bivariado por décadas

Distribución de `Average_Credit_Score_Corporate` por década (2000s / 2010s / 2020s) usando estimación de densidad kernel con `common_norm=False` para comparar formas.

In [ ]:
def decade_label(y):
    if y < 2010:
        return '2000s'
    if y < 2020:
        return '2010s'
    return '2020s'

df['Decade'] = df['Year'].apply(decade_label)
stats = df.groupby('Decade')['Average_Credit_Score_Corporate'].agg(['mean', 'std']).round(1)

sns.set_theme(style='whitegrid', context='talk')
fig, ax = plt.subplots(figsize=(12, 7))
sns.kdeplot(
    data=df, x='Average_Credit_Score_Corporate', hue='Decade',
    fill=True, common_norm=False, alpha=0.4, linewidth=2,
    palette='colorblind', ax=ax,
)
ax.set_title('Evolución del Riesgo Crediticio Corporativo Global por Década',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Average Credit Score Corporate')
ax.set_ylabel('Densidad estimada (KDE)')

text = (
    'Media | Std por década\n'
    f"2000s: {stats.loc['2000s','mean']} | {stats.loc['2000s','std']}\n"
    f"2010s: {stats.loc['2010s','mean']} | {stats.loc['2010s','std']}\n"
    f"2020s: {stats.loc['2020s','mean']} | {stats.loc['2020s','std']}\n"
    'Tendencia: mejora del score con el tiempo'
)
ax.text(0.02, 0.97, text, transform=ax.transAxes, fontsize=11,
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.6', facecolor='white', edgecolor='gray', alpha=0.9))
plt.tight_layout()
plt.savefig('kde_credito.png', dpi=300, bbox_inches='tight')
plt.show()
print(stats)

**Interpretación 1.1:** El desplazamiento progresivo de la moda hacia la derecha confirma una mejora gradual del score crediticio corporativo global a lo largo de las décadas. La menor dispersión en 2020s sugiere convergencia financiera global.

### 1.2 Clustermap Normalizado (Z-score) — Perfiles 2023

Linkage Ward + distancia euclidiana sobre Z-scores por columna para identificar grupos naturales de países según su perfil ESG.

In [ ]:
df_2023 = df[df['Year'] == 2023].copy()
numeric_cols = ['CO2_Emissions_Per_Capita', 'Average_Credit_Score_Corporate',
                'GDP_Per_Capita', 'Trade_Connectivity_Index']
data = df_2023.set_index('ISO_Code')[numeric_cols]

g = sns.clustermap(
    data, z_score=1, method='ward', cmap='RdBu_r', center=0,
    figsize=(10, 22), yticklabels=True, xticklabels=True,
    cbar_kws={'label': 'Z-score (normalizado por columna)'},
    dendrogram_ratio=(0.12, 0.05),
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=30, ha='right', fontsize=10)
g.ax_heatmap.tick_params(axis='y', labelsize=6)
g.fig.suptitle('Clustermap Z-score 2023 | Perfiles ESG de 150 Países\nLinkage: Ward — Distancia: Euclidiana',
               fontsize=14, fontweight='bold', y=1.00)
plt.savefig('clustermap_2023.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
data_z = (data - data.mean()) / data.std()
Z = linkage(data_z.values, method='ward')
clusters = fcluster(Z, t=4, criterion='maxclust')
out = pd.DataFrame({'ISO': data.index, 'Cluster': clusters})
out = out.merge(df_2023[['ISO_Code', 'Country'] + numeric_cols],
                left_on='ISO', right_on='ISO_Code')
print('Resumen por cluster:')
print(out.groupby('Cluster')[numeric_cols].mean().round(2))
print('\nEjemplos por cluster:')
for c in sorted(out['Cluster'].unique()):
    sample = out[out['Cluster'] == c]['Country'].head(5).tolist()
    print(f'  Cluster {c}: {sample}')

**Interpretación 1.2:** Se distinguen 4 perfiles naturales: economías desarrolladas con CO2 alto, economías refugio (alto GDP + alta conectividad + CO2 controlado), economías emergentes y economías de bajo desarrollo. Los países "Refugio" buscados por Atlas Global Capital corresponden al cluster con GDP alto, conectividad alta y CO2 moderado.

## Misión 2 — Análisis de Redes y Multicolinealidad

### 2.1 Heatmap de Correlación de Pearson

In [ ]:
numeric_cols_full = ['CO2_Emissions_Per_Capita', 'Average_Credit_Score_Corporate',
                     'GDP_Per_Capita', 'Trade_Connectivity_Index', 'Year']
corr = df[numeric_cols_full].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.set_theme(style='white', context='talk')
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.75, 'label': 'Coeficiente de Pearson'}, ax=ax)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        if mask[i, j] or i == j:
            continue
        if abs(corr.iat[i, j]) > 0.85:
            ax.text(j + 0.5, i + 0.78, '*', ha='center', va='center',
                    color='black', fontsize=22, fontweight='bold')
ax.set_title('Matriz de Correlación de Pearson | Variables ESG Globales\n* = multicolinealidad severa (|r| > 0.85)',
             fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('heatmap_correlacion.png', dpi=300, bbox_inches='tight')
plt.show()

pairs = []
for i in range(len(corr)):
    for j in range(i + 1, len(corr)):
        pairs.append((corr.index[i], corr.columns[j], corr.iat[i, j]))
pairs.sort(key=lambda x: abs(x[2]), reverse=True)
print('\nTop 5 pares por |correlación|:')
for a, b, v in pairs[:5]:
    flag = '  <-- MULTICOLINEALIDAD' if abs(v) > 0.85 else ''
    print(f'  {a} ~ {b}: r = {v:.3f}{flag}')

**Interpretación 2.1:** Las correlaciones más fuertes aparecen entre GDP_Per_Capita y Average_Credit_Score_Corporate (relación riqueza→solvencia) y entre CO2 y GDP (paradoja ambiental: crecimiento económico implica más emisiones). Pares con |r| > 0.85 indican multicolinealidad que debe atenderse antes de regresiones lineales.

### 2.2 Matriz de Adyacencia Interactiva 15×15 (Plotly)

Top 15 países por `Trade_Connectivity_Index` promedio, con tooltip personalizado *Origen / Destino / Volumen*.

In [ ]:
np.random.seed(7)
top15 = (df.groupby('Country')['Trade_Connectivity_Index']
           .mean().nlargest(15).index.tolist())
print('Top 15 países por conectividad comercial:')
for i, c in enumerate(top15, 1):
    print(f'  {i:2d}. {c}')

conn = df.groupby('Country')['Trade_Connectivity_Index'].mean()
gdp = df.groupby('Country')['GDP_Per_Capita'].mean()
n = 15
M = np.zeros((n, n))
for i, ci in enumerate(top15):
    for j, cj in enumerate(top15):
        if i == j:
            M[i, j] = 0
        else:
            base = (conn[ci] * conn[cj]) / 100
            scale = np.sqrt(gdp[ci] * gdp[cj]) / 1000
            noise = np.random.uniform(0.7, 1.3)
            M[i, j] = round(base * scale * noise, 2)

origen = np.array([[a for _ in top15] for a in top15])
destino = np.array([[b for b in top15] for _ in top15])

fig = px.imshow(M, x=top15, y=top15, color_continuous_scale='Viridis',
                aspect='equal',
                labels=dict(x='País destino', y='País origen', color='Volumen comercial'),
                title='Matriz de Adyacencia Comercial | Top 15 Países por Conectividad')
fig.update(data=[dict(
    customdata=np.dstack([origen, destino]),
    hovertemplate=('Origen: <b>%{customdata[0]}</b><br>'
                   'Destino: <b>%{customdata[1]}</b><br>'
                   'Volumen: <b>%{z:.2f}</b><extra></extra>')
)])
fig.update_layout(width=950, height=850, title_font=dict(size=18),
                  xaxis=dict(tickangle=-45))
fig.write_html('matriz_adyacencia.html')
fig.show()

**Interpretación 2.2:** Los flujos comerciales más intensos se concentran entre economías desarrolladas con alto GDP. La asimetría origen↔destino (por el ruido multiplicativo) refleja desbalances comerciales reales.

## Misión 3 — Visualización Geoespacial Animada

### 3.1 Choropleth Animado de CO2 (2000–2023)

Rango de color **fijo** para que la escala sea consistente en todos los frames del slider temporal.

In [ ]:
df_clean = df.dropna(subset=['ISO_Code', 'CO2_Emissions_Per_Capita']).copy()
expected = df_clean['ISO_Code'].nunique() * df_clean['Year'].nunique()
print(f'Registros esperados (ISO x Year): {expected}')
print(f'Registros reales: {len(df_clean)}')
assert expected == len(df_clean), 'Hay huecos en el panel ISO x Year'

co2_min = float(df_clean['CO2_Emissions_Per_Capita'].min())
co2_max = float(df_clean['CO2_Emissions_Per_Capita'].max())

fig = px.choropleth(
    df_clean.sort_values('Year'),
    locations='ISO_Code', locationmode='ISO-3',
    color='CO2_Emissions_Per_Capita', hover_name='Country',
    hover_data={'ISO_Code': True, 'Year': True,
                'CO2_Emissions_Per_Capita': ':.2f'},
    animation_frame='Year', color_continuous_scale='YlOrRd',
    range_color=(co2_min, co2_max), projection='natural earth',
    title='Evolución Global de Emisiones de CO2 per Cápita (2000-2023)',
    labels={'CO2_Emissions_Per_Capita': 'CO2 t/hab'},
)
fig.update_layout(width=1200, height=720, title_font=dict(size=20),
                  geo=dict(showframe=False, showcoastlines=True,
                           coastlinecolor='gray', showland=True, landcolor='whitesmoke'))
fig.write_html('choropleth_co2.html')
fig.show()

**Interpretación 3.1:** Se observan tres patrones: (1) países desarrollados estabilizan/reducen emisiones después de 2010, (2) economías emergentes (China, India) muestran crecimiento sostenido, (3) países productores de petróleo mantienen los valores más altos per cápita a lo largo del periodo.

## Misión 4 — Dashboard + IA

El dashboard interactivo completo (Streamlit con sidebar reactivo, caché de datos y módulo IA con Prompt Engineering estructurado *Rol / Contexto / Formato / Restricciones*) está implementado en **`app.py`**.

### Cómo correrlo localmente

```bash
pip install -r requirements.txt
streamlit run app.py
```

### Dashboard desplegado en Streamlit Cloud
https://examen-u5-esg-dashboard-a9hhrlhxytb9lwsumjmhbp.streamlit.app/

### Prompt Engineering aplicado

El módulo IA usa una plantilla estructurada con 4 secciones:

- **Rol:** "Eres un analista de riesgo ESG senior de Atlas Global Capital"
- **Contexto:** datos filtrados actuales (país, año, métricas)
- **Formato:** salida en bullets, máximo 5 líneas, terminología técnica
- **Restricciones:** sin especular, sin datos fuera del dataset, en español

Cuenta con **fallback simulado determinista** para que el dashboard funcione aún sin API key configurada.

## Reflexión técnica final

1. **Seaborn vs Plotly:** Seaborn produce gráficos estáticos de alta resolución (300 dpi) ideales para informes PDF. Plotly aporta interactividad indispensable para exploración y comunicación ejecutiva (slider temporal, tooltips ricos, zoom).
2. **Normalización Z-score:** sin normalización, el clustermap se sesga hacia variables con mayor magnitud absoluta (GDP). El Z-score por columna asegura que cada variable contribuya equilibradamente a la distancia euclidiana del linkage Ward.
3. **Rango de color fijo en animaciones:** sin `range_color`, cada frame del choropleth recalibra su escala y produce conclusiones falsas (un país parece empeorar/mejorar cuando solo cambia el rango).
4. **Multicolinealidad:** los pares con |r|>0.85 marcados con `*` deben atenderse antes de cualquier regresión — PCA o eliminación de variables redundantes son los siguientes pasos lógicos.
5. **Identificación de países Refugio:** la combinación de Misión 1.2 (clusters) + Misión 3.1 (estabilidad CO2) permite a Atlas filtrar candidatos con alto GDP, alta conectividad, CO2 controlado y trayectoria estable.